# Research Mode

Research mode is a behavioral pattern where the agent's primary activity is information gathering, synthesis, and citation. The agent explores a topic across multiple sources, evaluates source credibility, identifies patterns and contradictions, and produces a structured summary of findings.

The defining characteristic of research mode is breadth before depth: the agent casts a wide net across diverse sources before synthesizing conclusions. This prevents premature convergence on the first plausible answer and ensures the final synthesis reflects the actual state of knowledge rather than cherry-picked evidence.

This mode is classified as information gathering and synthesis. It can combine with any autonomy level:
- Research + copilot: Agent gathers information, human writes the synthesis.
- Research + supervised: Agent completes research phases, human approves before delivery.
- Research + fully autonomous: Agent gathers, synthesizes, and delivers the report end-to-end.

In [1]:
import os
import json
from dataclasses import dataclass, field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

### Initialize the language model
Research mode uses `temperature=0` to ensure consistent, factual outputs. When the agent's job is to accurately represent source material and identify contradictions, variability in phrasing is a liability - we want deterministic, repeatable synthesis across runs.

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)

print("LLM initialized for research mode.")

LLM initialized for research mode.


## Data structures: Sources and reports
Research mode requires more structure than other behavioral modes because traceability is a first-class concern. Every claim in the final report must be linkable to the specific source it came from, and every source must carry a credibility score so readers can weigh the evidence behind each claim.

Two dataclasses capture this structure. `Source` represents a single retrieved document - its location, its content excerpt, and its credibility score. `ResearchReport` is the accumulator for an entire research session: it holds the queries that were generated, all sources before and after filtering, the extracted claims, any contradictions found, the final synthesis, and an overall confidence level. Keeping all of this together in one object means any step in the pipeline can inspect the full research history, not just its own outputs.

In [3]:
@dataclass
class Source:
    """A single retrieved source document."""
    url: str                        # identifier used for deduplication across queries
    title: str                      # human-readable name for citation display
    snippet: str                    # the content excerpt that the agent will analyze
    credibility_score: float = 0.5  # reliability from 0.0 (low) to 1.0 (high)


@dataclass
class ResearchReport:
    """The complete output of a research session."""
    topic: str
    queries: list = field(default_factory=list)            # search queries generated to cover the topic
    all_sources: list = field(default_factory=list)        # every source retrieved, before filtering
    credible_sources: list = field(default_factory=list)   # subset that passed the credibility threshold
    claims: list = field(default_factory=list)             # factual claims extracted from credible sources
    contradictions: list = field(default_factory=list)     # conflicting claims found across sources
    synthesis: str = ""                                    # final narrative with citations
    confidence: str = "medium"                             # 'high', 'medium', or 'low'

The separation between `all_sources` and `credible_sources` is deliberate. Keeping all retrieved sources in the report allows retrospective analysis . a consumer can ask "why did the agent ignore this source?" - while the credible subset is what actually informs claims and synthesis. A report that silently discards sources without record is harder to audit and easier to manipulate by framing the search to exclude inconvenient evidence.

The `confidence` field is set by the pipeline at the end of the run based on how many credible sources were found. It is not a score produced by the LLM - it is a deterministic signal derived from the retrieval outcome, which makes it more reliable than asking the model to self-assess its own certainty.

## Mock search database
In a production research agent, the retrieval step would call an external search API - Tavily, Bing, Brave, or a proprietary document store. Here we use a static mock database that simulates a realistic mix of sources: high-credibility academic papers and institutional reports alongside one low-credibility source that the filtering step should reject.

Including the low-credibility source is intentional. If every source in the database passed the filter, the credibility step would appear to work without actually demonstrating anything. The mock database must contain at least one source the filter rejects - otherwise there is nothing to filter.

In [4]:
MOCK_SOURCE_DB = [
    Source(
        url="https://arxiv.org/abs/2303.08774",
        title="GPT-4 Technical Report",
        snippet="GPT-4 is a large multimodal model that accepts image and text inputs and produces "
                "text outputs. It exhibits human-level performance on various professional and academic benchmarks.",
        credibility_score=0.95
    ),
    Source(
        url="https://arxiv.org/abs/2210.11610",
        title="ReAct: Synergizing Reasoning and Acting in Language Models",
        snippet="We propose ReAct, a paradigm that synergizes reasoning and acting in LLMs. "
                "The model generates verbal reasoning traces and text actions in an interleaved manner, "
                "enabling dynamic planning and tool use.",
        credibility_score=0.93
    ),
    Source(
        url="https://lilianweng.github.io/posts/2023-06-23-agent/",
        title="LLM Powered Autonomous Agents — Lilian Weng",
        snippet="Building agents with LLM as its core controller is a cool concept. Several proof-of-concepts "
                "demos, such as AutoGPT and GPT-Engineer, are inspiring. An LLM agent consists of planning, "
                "memory, and tool use.",
        credibility_score=0.85
    ),
    Source(
        url="https://proceedings.neurips.cc/paper_files/paper/2022/toolformer",
        title="Toolformer: Language Models Can Teach Themselves to Use Tools",
        snippet="We introduce Toolformer, a model trained to decide which APIs to call, when to call them, "
                "what arguments to pass, and how to best incorporate the results into future token prediction.",
        credibility_score=0.91
    ),
    Source(
        url="https://random-blog.io/ai-agents-will-replace-everything",
        title="AI Agents Are Going To Replace Every Job By Next Year",
        snippet="AI agents can already do literally anything a human can do. Every job will be automated within "
                "12 months. Companies that don't adopt AI agents immediately will cease to exist.",
        credibility_score=0.15   # deliberately low — sensational, unattributed claims
    ),
    Source(
        url="https://openai.com/research/practices-for-governing-ai",
        title="Practices for Governing Agentic AI Systems — OpenAI",
        snippet="As AI systems become more capable of autonomous action, governance frameworks must address "
                "new challenges: delegation of authority, accountability gaps, and irreversibility of agent actions.",
        credibility_score=0.88
    ),
]

# Threshold below which a source is excluded from claim extraction and synthesis
CREDIBILITY_THRESHOLD = 0.60


def mock_search(query: str) -> list:
    """Simulate a search by returning the full mock database regardless of query.

    In production, this would call a real search API and return query-specific results.
    The deduplication step in the pipeline prevents the same source from being added twice
    when multiple queries all return the same mock database.
    """
    return MOCK_SOURCE_DB


print(f"Mock database: {len(MOCK_SOURCE_DB)} sources total")
print(f"  Pass threshold (>= {CREDIBILITY_THRESHOLD}): "
      f"{sum(1 for s in MOCK_SOURCE_DB if s.credibility_score >= CREDIBILITY_THRESHOLD)}")
print(f"  Below threshold: "
      f"{sum(1 for s in MOCK_SOURCE_DB if s.credibility_score < CREDIBILITY_THRESHOLD)}")

Mock database: 6 sources total
  Pass threshold (>= 0.6): 5
  Below threshold: 1


## Phase 1: Query generation
Before the agent retrieves a single document, it generates a set of diverse search queries. This step might seem like a minor implementation detail - why not just search the topic directly? - but it is where coverage bias is prevented. A single query filters the world through one framing of the question. Four queries covering distinct angles force the retrieval step to encounter perspectives it might not have found from a single starting point: technical foundations, recent developments, practical applications, and known limitations are often discussed in different documents by different authors.

The queries are generated once before any retrieval happens, and the resulting list is treated as a fixed plan. Generating queries on the fly - one per retrieval round - would allow earlier results to bias the framing of later queries, which is exactly the kind of confirmation bias the multi-query approach is designed to prevent.

In [5]:
def generate_research_queries(topic: str, scope: str = "") -> list:
    """Generate diverse search queries to ensure comprehensive topic coverage.

    Args:
        topic: The research question or subject.
        scope: Optional constraints (e.g., 'last 2 years', 'enterprise use cases').

    Returns:
        List of query strings covering different angles of the topic.
    """
    system_prompt = """You are a research librarian. Generate exactly 4 search queries for the given topic.

The 4 queries must cover different angles:
1. Technical foundations and core concepts
2. Recent developments and state of the art
3. Practical applications and real-world use
4. Limitations, challenges, or open problems

Respond with a JSON array of 4 strings. No markdown, no commentary:
["query 1", "query 2", "query 3", "query 4"]"""

    # Append scope only when it was provided — an empty scope string would add a noisy blank line
    prompt = f"Topic: {topic}" + (f"\nScope: {scope}" if scope else "")

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        # Expect a bare JSON array — no markdown fences, no surrounding explanation
        return json.loads(response.content)
    except json.JSONDecodeError:
        # Fall back to a single query using the raw topic so retrieval can still proceed
        return [topic]

The function asks the LLM to produce a JSON array directly rather than prose, which removes any need to parse natural language from the response. The four-angle structure in the system prompt - foundations, recent developments, applications, limitations - is the key to coverage: each angle targets a different slice of the literature and tends to surface different documents even when searching the same database.

The `scope` parameter is appended conditionally so an empty value does not produce a blank `Scope:` line that might confuse the model. The fallback on JSON parse failure returns the raw topic as a single query, ensuring the pipeline always has at least one query to work with.

In [6]:
# Run just the query generation step to see the four angles before any retrieval begins
topic = "LLM-based autonomous agents: capabilities, architectures, and limitations"
queries = generate_research_queries(topic)

print(f"Generated {len(queries)} queries for:\n  '{topic}'")
print()
for i, q in enumerate(queries, 1):
    print(f"  {i}. {q}")

Generated 4 queries for:
  'LLM-based autonomous agents: capabilities, architectures, and limitations'

  1. Technical foundations of LLM-based autonomous agents and their core architectures
  2. Recent advancements in LLM-based autonomous agents and state-of-the-art techniques
  3. Practical applications of LLM-based autonomous agents in various industries
  4. Limitations and challenges faced by LLM-based autonomous agents in real-world scenarios


## Phase 2: Credibility filtering
Not every source a search engine returns deserves equal weight in the final synthesis. A peer-reviewed paper and a viral blog post may make the same surface-level claim with very different levels of evidentiary support. The credibility filter enforces a minimum standard: sources that fall below the threshold are excluded from claim extraction and synthesis, though they remain visible in `ResearchReport.all_sources` as a record of what was encountered and rejected.

In this notebook, credibility scores are pre-assigned to each mock source to represent their realistic standing - academic papers from arXiv and NeurIPS score near 0.9, expert blogs near 0.85, and the clearly sensational blog post scores 0.15. In a production system, scoring would be a separate upstream step where the LLM evaluates the publisher, authorship, citation practice, and content quality of each retrieved document before it reaches this filter. The filter itself - the comparison against the threshold - stays exactly as shown here regardless of how the scores are computed.

In [7]:
def filter_credible_sources(sources: list, threshold: float = CREDIBILITY_THRESHOLD) -> list:
    """Filter sources below the credibility threshold and report the filter decision.

    Args:
        sources: All retrieved Source objects.
        threshold: Minimum credibility score to retain a source.

    Returns:
        Sources whose credibility_score meets or exceeds the threshold.
    """
    # Keep only sources that clear the threshold — direct comparison, no wrapper function
    credible = [s for s in sources if s.credibility_score >= threshold]

    # Print every source with a pass/fail icon so the filter decision is fully visible
    print(f"Credibility filter: {len(sources)} sources → {len(credible)} passed (threshold: {threshold})")
    for s in sources:
        icon = "\u2713" if s.credibility_score >= threshold else "\u2717"  # ✓ or ✗
        print(f"  {icon} [{s.credibility_score:.2f}] {s.title}")

    return credible

The comparison is performed directly in the list comprehension rather than through an intermediate scoring function. The scoring function pattern - a wrapper that simply reads a pre-assigned field - adds an indirection layer that obscures what the code is doing. The filter logic belongs here; the score computation belongs upstream, before the source reaches this function.

The print loop inside the function is intentional. In a research pipeline, transparency about which sources were excluded is as important as the list that passed. A consumer of the report should be able to verify the filter decision without re-running the pipeline, which means the rejection list must appear in the output - not be silently discarded.

In [8]:
# Apply the filter to the full mock database to see which sources pass
credible_sources = filter_credible_sources(MOCK_SOURCE_DB)

print(f"\n{len(credible_sources)} of {len(MOCK_SOURCE_DB)} sources will be used for claim extraction.")

Credibility filter: 6 sources → 5 passed (threshold: 0.6)
  ✓ [0.95] GPT-4 Technical Report
  ✓ [0.93] ReAct: Synergizing Reasoning and Acting in Language Models
  ✓ [0.85] LLM Powered Autonomous Agents — Lilian Weng
  ✓ [0.91] Toolformer: Language Models Can Teach Themselves to Use Tools
  ✗ [0.15] AI Agents Are Going To Replace Every Job By Next Year
  ✓ [0.88] Practices for Governing Agentic AI Systems — OpenAI

5 of 6 sources will be used for claim extraction.


## Phase 3: Claim extraction
Once credible sources have been identified, the agent reads each one and extracts a small number of specific, citable claims. This step converts source content - which may be long and cover multiple topics - into a structured list of individual statements that can be compared, cross-referenced, and cited in the final synthesis.

The key discipline here is attribution: each extracted claim must embed a citation that ties it back to its specific source. A claim that says "LLMs exhibit human-level performance on academic benchmarks" is useful in a synthesis; the same claim without a citation is indistinguishable from a hallucination. By embedding the citation directly in the claim string - rather than tracking it as a separate foreign key - the attribution travels with the claim regardless of how it is sorted, filtered, or merged downstream.

In [9]:
def extract_claims(source: Source) -> list:
    """Extract key factual claims from a single source, each ending with a citation.

    Args:
        source: The source to extract claims from.

    Returns:
        List of claim strings, each ending with '[Source: Title]'.
    """
    system_prompt = """You are a fact extractor. Extract 2-3 key factual claims from the source snippet.

Each claim must be:
- A complete, standalone sentence (understandable without reading the full source)
- Factual, not an opinion
- Ending with the source citation in square brackets: [Source: Title]

Respond with a JSON array of claim strings. No markdown, no commentary:
["Claim one [Source: Title]", "Claim two [Source: Title]"]"""

    prompt = f"Source title: {source.title}\nContent: {source.snippet}"

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        return json.loads(response.content)
    except json.JSONDecodeError:
        # Fall back to a minimal citation constructed directly from the snippet
        return [f"{source.snippet[:100]}... [Source: {source.title}]"]

The citation format `[Source: Title]` is embedded in each claim string rather than stored separately. This makes the extraction output self-contained: any downstream step - contradiction detection, synthesis, or a human reader reviewing the list - can see the attribution without joining against another data structure.

The fallback on JSON parse failure constructs a minimal claim by taking the first 100 characters of the snippet and appending a citation. This produces something usable rather than silently dropping the source — a source that was deemed credible should contribute at least one claim to the pool, even if the LLM's response was malformed.

In [10]:
# Extract claims from a single source to see the format before running on all credible sources
sample_source = MOCK_SOURCE_DB[0]   # GPT-4 Technical Report
sample_claims = extract_claims(sample_source)

print(f"Claims extracted from: {sample_source.title}")
print()
for c in sample_claims:
    print(f"  - {c}")

Claims extracted from: GPT-4 Technical Report

  - GPT-4 is a large multimodal model that accepts image and text inputs and produces text outputs. [Source: GPT-4 Technical Report]
  - GPT-4 exhibits human-level performance on various professional and academic benchmarks. [Source: GPT-4 Technical Report]


## Phase 4: Contradiction detection
After claim extraction is complete, the agent holds a pool of statements drawn from multiple independent sources. Before synthesizing these into a narrative, it checks whether any two claims directly conflict with each other. This step is one of the most important differentiators between research mode and simple summarization.

A summarizer that encounters contradictory sources will typically do one of two things: silently favor one source over another based on the order it was processed, or merge the conflicting claims into a vague middle-ground statement that misrepresents both. A research agent must do neither. It must surface the conflict explicitly - naming both positions and explaining why they conflict - so the reader can weigh the disagreement with full information. A report that hides contradictions is not objective synthesis; it is advocacy dressed as research.

In [11]:
def detect_contradictions(all_claims: list) -> list:
    """Identify claims across sources that directly contradict each other.

    Args:
        all_claims: Combined list of claims from all credible sources.

    Returns:
        List of contradiction dicts with 'claim_a', 'claim_b', and 'reason' keys.
        Empty list if no contradictions are found.
    """
    # A contradiction requires at least two claims — a single claim can never conflict with itself
    if len(all_claims) < 2:
        return []

    system_prompt = """You are a fact-checker looking for contradictions.

Review the list of claims and identify any that directly contradict each other.
Only flag genuine factual contradictions — not differences in emphasis or scope.

Respond with a JSON array. If no contradictions exist, return an empty array [].
For each contradiction found:
{"claim_a": "...", "claim_b": "...", "reason": "why they conflict"}"""

    # Format claims as a bulleted list so the LLM can scan them clearly
    claims_text = "\n".join(f"- {c}" for c in all_claims)
    prompt = f"Claims to review:\n{claims_text}"

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        result = json.loads(response.content)
        # Guard against the model returning a single dict instead of a list when only one contradiction is found
        return result if isinstance(result, list) else [result]
    except json.JSONDecodeError:
        return []

The early-return guard on claim count is a correctness check, not a performance optimization - a contradiction requires at least two parties. The `isinstance(result, list)` check handles a common model failure mode: when the system prompt says "return an empty array `[]`", models sometimes interpret this as a format hint for the empty case only and return a bare dict when exactly one contradiction is found. Wrapping a non-list result in a list makes the downstream synthesis step safe to iterate without an `if isinstance` check there.

The instruction to flag only "genuine factual contradictions" - not differences in emphasis or scope - is important for precision. Two sources that describe the same phenomenon at different levels of detail are not contradicting each other; only claims that cannot both be true at the same time qualify.

## Phase 5: Synthesis
Synthesis is the step where the agent converts a collection of extracted, credibility-filtered, and contradiction-checked claims into a coherent narrative. It is where research mode most clearly distinguishes itself from aggregation: the agent must organize claims into a logical structure, weave in citations for every factual statement, acknowledge where contradictions or gaps exist, and produce conclusions that are grounded in the evidence.

The synthesis function receives the full `ResearchReport` object rather than individual arguments because it needs access to three related collections simultaneously - the extracted claims, the credible sources for citation formatting, and the detected contradictions. Passing these as separate arguments would require callers to unpack the report, which adds surface area without benefit. The synthesis step is always the last phase, so it can safely assume the report is fully populated.

In [12]:
def synthesize_findings(report: ResearchReport) -> str:
    """Synthesize all claims into a structured research narrative with citations.

    Args:
        report: A fully populated ResearchReport with claims and credible_sources.

    Returns:
        Structured narrative string with Executive Summary, Key Findings,
        Contradictions, Gaps, and References sections.
    """
    # Format claims as a bulleted list for the synthesis prompt
    claims_text = "\n".join(f"- {c}" for c in report.claims)

    # Number each source for citation references in the report
    sources_text = "\n".join(
        f"[{i+1}] {s.title} ({s.url}) — credibility: {s.credibility_score}"
        for i, s in enumerate(report.credible_sources)
    )

    # Provide 'None found.' as a placeholder so the synthesis prompt always has content for that section
    contradictions_text = (
        "\n".join(
            f"- {c['claim_a']} vs {c['claim_b']}: {c['reason']}"
            for c in report.contradictions
        )
        if report.contradictions else "None found."
    )

    system_prompt = """You are a research analyst. Synthesize the provided claims into a structured report.

Structure:
## Executive Summary
[3-4 sentences covering the most important findings]

## Key Findings
[Organized narrative with inline citations like [Source: Title]]

## Contradictions or Disputes
[Any conflicting information — do not hide this]

## Gaps and Limitations
[What remains unknown or uncertain]

## References
[Numbered list of sources]

Rules: Cite sources for every factual claim. Acknowledge uncertainty explicitly. Do not invent claims."""

    prompt = f"""Research topic: {report.topic}

Extracted claims:
{claims_text}

Contradictions identified:
{contradictions_text}

Sources:
{sources_text}"""

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])
    return response.content.strip()

The `contradictions_text` fallback to `'None found.'` is not merely cosmetic. The synthesis system prompt instructs the model to include a Contradictions section regardless of whether any were found. Supplying a placeholder keeps the structured output format intact and signals explicitly to the model - and to the reader - that contradiction detection ran and found nothing, rather than that it was skipped.

The source numbering (`[{i+1}] {s.title}`) creates a correspondence between inline citations and the References section at the end of the report. The model is not guaranteed to honor this numbering strictly, but providing it gives the synthesis prompt a reference format that is easier to follow than asking the model to invent its own citation scheme.

## Complete research pipeline
With all five phase functions in place, the pipeline function assembles them into a single end-to-end run. It takes a topic and an optional scope constraint, and returns a fully populated `ResearchReport`. The function logs its progress at each phase so the run is transparent - a researcher watching the pipeline execute can see which queries were generated, which sources passed or failed the filter, and how many claims were extracted before the synthesis begins.

One detail worth noting is the URL-based deduplication in the retrieval phase. Because the mock search function returns the same database for every query, and because in production different queries often surface the same documents, the pipeline must track which source URLs have already been added and skip duplicates. Without this step, the claims pool would contain multiple copies of the same text, inflating the apparent coverage of some sources.

In [13]:
def conduct_research(topic: str, scope: str = "") -> ResearchReport:
    """Run the complete five-phase research pipeline for a given topic.

    Args:
        topic: The research question or topic to investigate.
        scope: Optional scope constraints (e.g., 'last 2 years only').

    Returns:
        A fully populated ResearchReport.
    """
    report = ResearchReport(topic=topic)   # empty report that will accumulate results across phases

    print(f"\nRESEARCH: {topic}")
    print("=" * 60)

    # Phase 1: Generate diverse queries before retrieving any sources
    print("\n[1/5] Generating research queries...")
    report.queries = generate_research_queries(topic, scope)
    for q in report.queries:
        print(f"  - {q}")

    # Phase 2: Retrieve sources for each query and deduplicate by URL
    print("\n[2/5] Retrieving sources...")
    seen_urls = set()   # URL set prevents the same source from being added more than once across queries
    for query in report.queries:
        for source in mock_search(query):
            if source.url not in seen_urls:
                seen_urls.add(source.url)
                report.all_sources.append(source)
    print(f"  Retrieved {len(report.all_sources)} unique sources across all queries.")

    # Phase 3: Apply credibility filter — only credible sources proceed to claim extraction
    print("\n[3/5] Filtering by credibility...")
    report.credible_sources = filter_credible_sources(report.all_sources)

    if len(report.credible_sources) < 2:
        # Fewer than 2 credible sources is insufficient for confident cross-source conclusions
        print("  WARNING: Fewer than 2 credible sources found. Confidence will be LOW.")
        report.confidence = "low"

    # Phase 4: Extract citable claims from each credible source
    print("\n[4/5] Extracting claims from credible sources...")
    for source in report.credible_sources:
        claims = extract_claims(source)
        report.claims.extend(claims)   # accumulate into a flat list across all sources
        print(f"  {source.title}: {len(claims)} claims extracted")

    # Check the full claim pool for contradictions before synthesizing
    report.contradictions = detect_contradictions(report.claims)
    if report.contradictions:
        print(f"  \u26a0 {len(report.contradictions)} contradiction(s) detected")

    # Phase 5: Synthesize all claims into a structured narrative
    print("\n[5/5] Synthesizing findings...")
    report.synthesis = synthesize_findings(report)

    # Assign confidence based on source count — 4+ credible sources → high, fewer → medium
    if report.confidence != "low":
        report.confidence = "high" if len(report.credible_sources) >= 4 else "medium"

    return report

In [14]:
# Run the full pipeline on the same topic used for the query generation demo
report = conduct_research(
    topic="LLM-based autonomous agents: capabilities, architectures, and limitations"
)

print("\n" + "=" * 60)
print("RESEARCH REPORT")
print("=" * 60)
print(f"Topic      : {report.topic}")
print(f"Sources    : {len(report.credible_sources)} credible (of {len(report.all_sources)} total)")
print(f"Claims     : {len(report.claims)}")
print(f"Confidence : {report.confidence.upper()}")
print()
print(report.synthesis)


RESEARCH: LLM-based autonomous agents: capabilities, architectures, and limitations

[1/5] Generating research queries...
  - Technical foundations of LLM-based autonomous agents and their core architectures
  - Recent advancements in LLM-based autonomous agents and state-of-the-art techniques
  - Practical applications of LLM-based autonomous agents in various industries
  - Limitations and challenges faced by LLM-based autonomous agents in real-world scenarios

[2/5] Retrieving sources...
  Retrieved 6 unique sources across all queries.

[3/5] Filtering by credibility...
Credibility filter: 6 sources → 5 passed (threshold: 0.6)
  ✓ [0.95] GPT-4 Technical Report
  ✓ [0.93] ReAct: Synergizing Reasoning and Acting in Language Models
  ✓ [0.85] LLM Powered Autonomous Agents — Lilian Weng
  ✓ [0.91] Toolformer: Language Models Can Teach Themselves to Use Tools
  ✗ [0.15] AI Agents Are Going To Replace Every Job By Next Year
  ✓ [0.88] Practices for Governing Agentic AI Systems — OpenAI



## Handling sparse sources
Real research frequently encounters situations where very few credible sources can be found on a topic - either because the topic is niche, the search database is limited, or the credibility threshold is set high. The naive response is to synthesize from whatever is available and present the result as if it were fully grounded. The honest response is to acknowledge the limitation explicitly before any conclusions are drawn.

A report that says "only 1 credible source was found; the following conclusions are speculative" is far more valuable than one that presents single-source findings with the same confidence as multi-source findings. Transparent limitations build trust in the reports that do have strong evidence, because readers know the system does not overstate what it knows.

In [15]:
def handle_sparse_sources(topic: str, found_sources: list, min_sources: int = 3) -> str:
    """Generate an honest disclosure report when credible sources are insufficient.

    Args:
        topic: The research topic that was investigated.
        found_sources: The credible sources that were found.
        min_sources: The minimum expected for confident conclusions.

    Returns:
        A disclosure string naming the source gap and recommending next steps.
    """
    # Build the source list inline so the disclosure is self-contained
    source_lines = "\n".join(
        f"  - {s.title} [credibility: {s.credibility_score}]" for s in found_sources
    )
    return (
        f"LIMITED RESEARCH — {topic}\n"
        f"Only {len(found_sources)} credible source(s) found (minimum expected: {min_sources}).\n\n"
        f"Available evidence:\n{source_lines}\n\n"
        f"Recommendation: Commission primary research or consult domain experts before "
        f"drawing conclusions from this limited evidence base."
    )


# Demonstrate with a scenario where only one high-credibility source was found
# Filter to sources above 0.9 and take only the first one — simulating a sparse result
sparse_sources = [s for s in MOCK_SOURCE_DB if s.credibility_score >= 0.9][:1]
print(handle_sparse_sources("Emerging AI governance frameworks", sparse_sources))

LIMITED RESEARCH — Emerging AI governance frameworks
Only 1 credible source(s) found (minimum expected: 3).

Available evidence:
  - GPT-4 Technical Report [credibility: 0.95]

Recommendation: Commission primary research or consult domain experts before drawing conclusions from this limited evidence base.


- **Breadth before depth**: Generating diverse queries across multiple angles before retrieving any documents prevents coverage bias. Querying from a single framing produces results that reflect that framing - not the actual state of knowledge on the topic.
- **Contradictions must be surfaced, not resolved silently**: If two credible sources disagree, the agent's job is to name both positions and explain the conflict - not to choose sides, average the positions, or suppress the discrepancy. A report that hides contradictions is advocacy, not research.
- **Synthesis is not aggregation**: Pasting extracted claims together is not a synthesis. A synthesis organizes claims into a logical structure, signals which conclusions have strong evidence and which are uncertain, and explicitly acknowledges what the research did not find.
- **Transparent limitations build trust**: A report that discloses "only 1 credible source was found" is more valuable than one that presents single-source findings at full confidence. Honest scarcity disclosure allows readers to calibrate their reliance on the report - and increases trust in the reports that do have strong evidence.